# ProtiQ on Google Colab

Protein function prediction from sequence and structure similarity. By Muyi Tang and Xenia Huang.

This notebook installs ProtiQ, downloads real training data from UniProt, trains and evaluates a model, and predicts the uncharacterized proteins of *Pseudomonas aeruginosa* PA14. Run the cells top to bottom. Steps 1 to 5 take about 5 minutes on a free Colab runtime.

## 1. Install

In [ ]:
# Replace YOUR-USERNAME after you push the repo to GitHub
!pip install -q git+https://github.com/muyit86macc/ProtiQ.git
!apt-get install -y -qq diamond-aligner > /dev/null
!protiq --version && diamond version

## 2. Download training data and targets from UniProt

In [ ]:
!protiq fetch --scheme virulence --per-class 3000 --out data/training_virulence.csv
!protiq fetch-targets --taxon 208963 --limit 400 --out data/targets_pa14.csv

## 3. Train and evaluate

The output compares ProtiQ to copying the top BLAST hit's label and to always guessing the majority class.

In [ ]:
!protiq train --data data/training_virulence.csv --backend diamond --out models/virulence.joblib

## 4. Reproduce the k-sweep and the ablation test

In [ ]:
!protiq sweep-k --data data/training_virulence.csv --backend diamond
!protiq ablate --data data/training_virulence.csv --backend diamond

## 5. Predict the uncharacterized PA14 proteins

In [ ]:
!protiq predict --model models/virulence.joblib --csv data/targets_pa14.csv --out results/pa14_virulence.csv
import pandas as pd
pred = pd.read_csv("results/pa14_virulence.csv")
supported = pred[~pred.evidence.str.startswith("no homologs")]
print(f"{len(supported)} of {len(pred)} predictions are backed by homologs")
supported.sort_values("confidence", ascending=False).head(15)

## 6. Explain one prediction with LIME

In [ ]:
from IPython.display import HTML
from protiq.model import load_bundle
from protiq.explain import explain_prediction, format_explanation

bundle = load_bundle("models/virulence.joblib")
row = pd.read_csv("data/targets_pa14.csv").set_index("uniprot_id").loc["A0A0H2ZCC1"]
result = explain_prediction(bundle, "A0A0H2ZCC1", row.sequence, html_path="lime.html")
print(format_explanation(result))
HTML(open("lime.html").read())

## 7. Optional: enzyme class model with AlphaFold features

`structure-batch` downloads each protein's AlphaFold model once, computes pLDDT and compactness features, and caches them.

In [ ]:
!protiq fetch --scheme ec --per-class 500 --out data/training_ec.csv
!protiq structure-batch --csv data/training_ec.csv
!protiq ablate --data data/training_ec.csv --backend diamond --alphafold

## 8. Next experiment: the Foldseek structural arm

This adds structural neighbor votes. Foldseek compares each protein's AlphaFold model to the reference proteins' models. It tests the original project's finding that 3D structure matters, and it's the best chance of characterizing proteins that have no sequence homologs. It needs the PDB files, so it downloads more data.

In [ ]:
!wget -q https://github.com/steineggerlab/foldseek/releases/download/10-941cd33/foldseek-linux-avx2.tar.gz && tar xzf foldseek-linux-avx2.tar.gz && cp foldseek/bin/foldseek /usr/local/bin/
!protiq structure-batch --csv data/training_virulence.csv --keep-pdb
!protiq ablate --data data/training_virulence.csv --backend diamond --foldseek --alphafold